In [0]:
"""
sim_purchase_delta.py
=====================
Simulation script: generates synthetic Purchase Orders + Lines
and writes them into Databricks Delta tables:

    smart_odoo.silver.purchase_order
    smart_odoo.silver.purchase_order_line

Rules
-----
- purchase_order_id   : 1 → 500
- purchase_order_line_id : sequential, scoped per order
- Date Start          : 2025-01-01
- purchase_state      : draft | sent | purchase | done | to_approve | cancel
- invoice_status      : to_invoice | invoiced | no_invoice
- partner_id          : 1 → 50
- product_id          : 1 → 100
"""

import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType,
    BooleanType, TimestampType
)

# ─────────────────────────────────────────────
# SPARK SESSION
# ─────────────────────────────────────────────
spark = SparkSession.builder.appName("sim_purchase_delta").getOrCreate()

USE_CATALOG   = "smart_odoo"
USE_SCHEMA    = "silver"
SOURCE_DB     = "python_sim"
TAX_RATE      = 0.14

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

# ─────────────────────────────────────────────
# REFERENCE LOOKUP TABLES  (in-memory simulation)
# In a real pipeline these would come from bronze tables.
# ─────────────────────────────────────────────

PARTNERS = {i: f"Supplier_{i:03d}"        for i in range(1, 51)}   # 1-50
PRODUCTS = {i: (f"Product_{i:03d}",
                round(random.uniform(10.0, 500.0), 2))              # id → (name, cost)
            for i in range(1, 101)}                                  # 1-100

COMPANIES       = {1: "Smart Odoo LLC"}
CURRENCIES      = {1: ("USD", 1.0), 2: ("EUR", 1.08), 3: ("EGP", 0.021)}
USERS           = {i: f"User_{i}" for i in range(1, 11)}
PAYMENT_TERMS   = {1: "Immediate Payment", 2: "Net 30", 3: "Net 60", 4: "2/10 Net 30"}
INCOTERMS       = {1: "EXW", 2: "FOB", 3: "CIF", 4: "DDP", None: None}
FISCAL_POS      = {1: "Domestic", 2: "EU", 3: "Export", None: None}
PICKING_TYPES   = {1: "Receipts", 2: "Returns"}
UOMS            = {1: "Units", 2: "kg", 3: "liters", 4: "pcs", 5: "boxes"}
PRIORITIES      = ["0", "1", "2", "3"]   # 0=normal, 1=urgent, etc.
RECEIPT_STATUS  = ["pending", "partial", "full", "nothing_to_receive"]

# ─────────────────────────────────────────────
# HELPER: random nullable foreign key
# ─────────────────────────────────────────────
def maybe(d: dict, none_prob: float = 0.15):
    """Return a random key from dict, or None with none_prob probability."""
    if random.random() < none_prob:
        return None
    return random.choice(list(d.keys()))


# ─────────────────────────────────────────────
# DATE ENGINE
# ─────────────────────────────────────────────
START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 4, 22)

def random_date() -> datetime:
    delta = (END_DATE - START_DATE).days
    return START_DATE + timedelta(days=random.randint(0, delta))

def generate_dates(state: str):
    date_order   = random_date()
    date_approve = date_order + timedelta(days=random.randint(0, 3))
    date_planned = date_approve + timedelta(days=random.randint(3, 20))

    if state == "done":
        effective_date = date_planned + timedelta(days=random.randint(1, 5))
    else:
        effective_date = None

    return date_order, date_approve, date_planned, effective_date


# ─────────────────────────────────────────────
# STATE ENGINE
# ─────────────────────────────────────────────
def generate_state() -> str:
    r = random.random()
    if r < 0.08:   return "draft"
    elif r < 0.16: return "sent"
    elif r < 0.30: return "purchase"
    elif r < 0.85: return "done"
    else:          return "cancel"

def generate_invoice_status(state: str) -> str:
    if state in ("draft", "sent", "cancel"):
        return "no"
    elif state == "purchase":
        return "to invoice"
    else:                          # done
        return "invoiced"

def generate_receipt_status(state: str) -> str:
    if state in ("draft", "sent", "cancel"):
        return "nothing_to_receive"
    elif state == "purchase":
        return random.choice(["pending", "partial"])
    else:
        return "full"


# ─────────────────────────────────────────────
# RESOLVE STARTING IDs  (append-safe)
# ─────────────────────────────────────────────
try:
    max_po_id = spark.sql(
        f"SELECT COALESCE(MAX(purchase_order_id), 0) FROM {USE_CATALOG}.{USE_SCHEMA}.purchase_order"
    ).collect()[0][0]
except Exception:
    max_po_id = 0   # table doesn't exist yet on first run

try:
    max_pol_id = spark.sql(
        f"SELECT COALESCE(MAX(purchase_order_line_id), 0) FROM {USE_CATALOG}.{USE_SCHEMA}.purchase_order_line"
    ).collect()[0][0]
except Exception:
    max_pol_id = 0

po_start  = max_po_id  + 1
pol_start = max_pol_id + 1

print(f"Starting purchase_order_id      from : {po_start}")
print(f"Starting purchase_order_line_id from : {pol_start}")


# ─────────────────────────────────────────────
# BUILD ROWS
# ─────────────────────────────────────────────
po_rows  = []
pol_rows = []

pol_id = pol_start   # global line counter

for po_id in range(po_start, po_start + 1000):   # always adds exactly 500 orders

    partner_id   = random.randint(1, 50)
    partner_name = PARTNERS[partner_id]

    company_id   = 1
    company_name = COMPANIES[1]

    cur_id       = random.choice([1, 2, 3])
    cur_name, cur_rate = CURRENCIES[cur_id]

    user_id      = random.choice(list(USERS.keys()))
    user_name    = USERS[user_id]

    pt_id        = random.choice(list(PAYMENT_TERMS.keys()))
    pt_name      = PAYMENT_TERMS[pt_id]

    inc_id       = maybe(INCOTERMS, none_prob=0.3)
    inc_name     = INCOTERMS.get(inc_id)

    fp_id        = maybe(FISCAL_POS, none_prob=0.4)
    fp_name      = FISCAL_POS.get(fp_id)

    pick_id      = random.choice(list(PICKING_TYPES.keys()))
    pick_name    = PICKING_TYPES[pick_id]

    state           = generate_state()
    invoice_status  = generate_invoice_status(state)
    rcpt_status     = generate_receipt_status(state)

    date_order, date_approve, date_planned, effective_date = generate_dates(state)
    now = datetime.now()

    order_name   = f"PO/{date_order.year}/{po_id:05d}"
    priority     = random.choice(PRIORITIES)
    origin       = f"RFQ-{random.randint(1000, 9999)}" if random.random() > 0.4 else None
    partner_ref  = f"REF-{random.randint(100, 999)}"   if random.random() > 0.5 else None
    note         = random.choice(["Urgent delivery", "Check quality", None, None, None])
    invoice_count = 1 if state == "done" else (0 if state in ("draft", "sent", "cancel") else random.randint(0, 1))
    locked        = state == "done" and random.random() > 0.3
    acknowledged  = state not in ("draft", "cancel")
    rcpt_reminder = random.random() > 0.7

    order_untaxed = 0.0
    order_tax     = 0.0

    # ── ORDER LINES ────────────────────────────
    num_lines = random.randint(1, 6)

    for _ in range(num_lines):

        prod_id              = random.randint(1, 100)
        prod_name, prod_cost = PRODUCTS[prod_id]

        uom_id   = random.choice(list(UOMS.keys()))
        uom_name = UOMS[uom_id]

        qty          = float(random.randint(30, 50))
        price_unit   = round(prod_cost * random.uniform(0.90, 1.20), 4)
        discount     = round(random.choice([0.0, 0.0, 0.0, 5.0, 10.0, 15.0]), 2)
        effective_pu = price_unit * (1 - discount / 100)

        subtotal  = round(qty * effective_pu, 2)
        price_tax = round(subtotal * TAX_RATE, 2)
        total     = round(subtotal + price_tax, 2)

        # received quantities
        if state == "done":
            qty_received = qty
        elif state == "purchase":
            qty_received = round(qty * random.uniform(0.5, 1.0), 2)
        else:
            qty_received = 0.0

        qty_invoiced      = qty_received if state == "done"     else 0.0
        qty_received_man  = 0.0
        qty_to_invoice    = max(0.0, qty_received - qty_invoiced)

        order_untaxed += subtotal
        order_tax     += price_tax

        line_name     = f"{prod_name} - {uom_name}"
        date_planned_line = date_planned + timedelta(days=random.randint(0, 5))

        pol_rows.append(Row(
            purchase_order_line_id      = pol_id,
            order_id                    = po_id,
            purchase_order_name         = order_name,
            product_id                  = prod_id,
            product_name                = prod_name,
            product_uom_id              = uom_id,
            product_uom_name            = uom_name,
            partner_id                  = partner_id,
            partner_name                = partner_name,
            company_id                  = company_id,
            company_name                = company_name,
            display_type                = None,
            order_line_name             = line_name,
            product_description_variants= None,
            qty_received_method         = "stock_moves" if state in ("purchase","done") else "manual",
            analytic_distribution       = None,
            product_qty                 = qty,
            product_uom_qty             = qty,
            discount                    = discount,
            price_unit                  = price_unit,
            price_subtotal              = subtotal,
            price_total                 = total,
            price_tax                   = price_tax,
            technical_price_unit        = price_unit,
            qty_invoiced                = qty_invoiced,
            qty_received                = qty_received,
            qty_received_manual         = qty_received_man,
            qty_to_invoice              = qty_to_invoice,
            is_downpayment              = False,
            propagate_cancel            = state == "cancel",
            date_planned                = date_planned_line,
            created_at                  = date_order,
            updated_at                  = date_order,
            dwh_loaded_at               = now,
            dwh_source_db               = SOURCE_DB,
        ))
        pol_id += 1

    # ── ORDER HEADER ───────────────────────────
    amount_total    = round(order_untaxed + order_tax, 2)
    amount_total_cc = round(amount_total * cur_rate, 2)

    po_rows.append(Row(
        purchase_order_id       = po_id,
        partner_id              = partner_id,
        partner_name            = partner_name,
        company_id              = company_id,
        company_name            = company_name,
        currency_id             = cur_id,
        currency_name           = cur_name,
        user_id                 = user_id,
        user_name               = user_name,
        payment_term_id         = pt_id,
        payment_term_name       = pt_name,
        incoterm_id             = inc_id,
        incoterm_name           = inc_name,
        fiscal_position_id      = fp_id,
        fiscal_position_name    = fp_name,
        picking_type_id         = pick_id,
        picking_type_name       = pick_name,
        invoice_count           = invoice_count,
        order_name              = order_name,
        priority                = priority,
        origin                  = origin,
        partner_ref             = partner_ref,
        order_state             = state,
        invoice_status          = invoice_status,
        receipt_status          = rcpt_status,
        note                    = note,
        amount_untaxed          = round(order_untaxed, 2),
        amount_tax              = round(order_tax, 2),
        amount_total            = amount_total,
        amount_total_cc         = amount_total_cc,
        currency_rate           = cur_rate,
        locked                  = locked,
        acknowledged            = acknowledged,
        receipt_reminder_email  = rcpt_reminder,
        date_order              = date_order,
        date_approve            = date_approve,
        date_planned            = date_planned,
        effective_date          = effective_date,
        created_at              = date_order,
        updated_at              = date_order,
        dwh_loaded_at           = now,
        dwh_source_db           = SOURCE_DB,
    ))


# ─────────────────────────────────────────────
# DEFINE SCHEMAS  (mirrors the Delta DDL exactly)
# ─────────────────────────────────────────────

po_schema = StructType([
    StructField("purchase_order_id",        IntegerType(),   True),
    StructField("partner_id",               IntegerType(),   True),
    StructField("partner_name",             StringType(),    True),
    StructField("company_id",               IntegerType(),   True),
    StructField("company_name",             StringType(),    True),
    StructField("currency_id",              IntegerType(),   True),
    StructField("currency_name",            StringType(),    True),
    StructField("user_id",                  IntegerType(),   True),
    StructField("user_name",                StringType(),    True),
    StructField("payment_term_id",          IntegerType(),   True),
    StructField("payment_term_name",        StringType(),    True),
    StructField("incoterm_id",              IntegerType(),   True),
    StructField("incoterm_name",            StringType(),    True),
    StructField("fiscal_position_id",       IntegerType(),   True),
    StructField("fiscal_position_name",     StringType(),    True),
    StructField("picking_type_id",          IntegerType(),   True),
    StructField("picking_type_name",        StringType(),    True),
    StructField("invoice_count",            IntegerType(),   True),
    StructField("order_name",               StringType(),    True),
    StructField("priority",                 StringType(),    True),
    StructField("origin",                   StringType(),    True),
    StructField("partner_ref",              StringType(),    True),
    StructField("order_state",              StringType(),    True),
    StructField("invoice_status",           StringType(),    True),
    StructField("receipt_status",           StringType(),    True),
    StructField("note",                     StringType(),    True),
    StructField("amount_untaxed",           DoubleType(),    True),
    StructField("amount_tax",               DoubleType(),    True),
    StructField("amount_total",             DoubleType(),    True),
    StructField("amount_total_cc",          DoubleType(),    True),
    StructField("currency_rate",            DoubleType(),    True),
    StructField("locked",                   BooleanType(),   True),
    StructField("acknowledged",             BooleanType(),   True),
    StructField("receipt_reminder_email",   BooleanType(),   True),
    StructField("date_order",               TimestampType(), True),
    StructField("date_approve",             TimestampType(), True),
    StructField("date_planned",             TimestampType(), True),
    StructField("effective_date",           TimestampType(), True),
    StructField("created_at",               TimestampType(), True),
    StructField("updated_at",               TimestampType(), True),
    StructField("dwh_loaded_at",            TimestampType(), True),
    StructField("dwh_source_db",            StringType(),    True),
])

pol_schema = StructType([
    StructField("purchase_order_line_id",        IntegerType(),   True),
    StructField("order_id",                      IntegerType(),   True),
    StructField("purchase_order_name",           StringType(),    True),
    StructField("product_id",                    IntegerType(),   True),
    StructField("product_name",                  StringType(),    True),
    StructField("product_uom_id",                IntegerType(),   True),
    StructField("product_uom_name",              StringType(),    True),
    StructField("partner_id",                    IntegerType(),   True),
    StructField("partner_name",                  StringType(),    True),
    StructField("company_id",                    IntegerType(),   True),
    StructField("company_name",                  StringType(),    True),
    StructField("display_type",                  StringType(),    True),
    StructField("order_line_name",               StringType(),    True),
    StructField("product_description_variants",  StringType(),    True),
    StructField("qty_received_method",           StringType(),    True),
    StructField("analytic_distribution",         StringType(),    True),
    StructField("product_qty",                   DoubleType(),    True),
    StructField("product_uom_qty",               DoubleType(),    True),
    StructField("discount",                      DoubleType(),    True),
    StructField("price_unit",                    DoubleType(),    True),
    StructField("price_subtotal",                DoubleType(),    True),
    StructField("price_total",                   DoubleType(),    True),
    StructField("price_tax",                     DoubleType(),    True),
    StructField("technical_price_unit",          DoubleType(),    True),
    StructField("qty_invoiced",                  DoubleType(),    True),
    StructField("qty_received",                  DoubleType(),    True),
    StructField("qty_received_manual",           DoubleType(),    True),
    StructField("qty_to_invoice",                DoubleType(),    True),
    StructField("is_downpayment",                BooleanType(),   True),
    StructField("propagate_cancel",              BooleanType(),   True),
    StructField("date_planned",                  TimestampType(), True),
    StructField("created_at",                    TimestampType(), True),
    StructField("updated_at",                    TimestampType(), True),
    StructField("dwh_loaded_at",                 TimestampType(), True),
    StructField("dwh_source_db",                 StringType(),    True),
])


# ─────────────────────────────────────────────
# CREATE DATAFRAMES
# ─────────────────────────────────────────────
df_po  = spark.createDataFrame(po_rows,  schema=po_schema)
df_pol = spark.createDataFrame(pol_rows, schema=pol_schema)

print(f"purchase_order rows      : {df_po.count()}")
print(f"purchase_order_line rows : {df_pol.count()}")


# ─────────────────────────────────────────────
# WRITE TO DELTA  (overwrite for simulation runs)
# ─────────────────────────────────────────────
(
    df_po
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{USE_CATALOG}.{USE_SCHEMA}.purchase_order")
)

(
    df_pol
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{USE_CATALOG}.{USE_SCHEMA}.purchase_order_line")
)


print("✅  DONE — Delta tables written to smart_odoo.silver")